
# Laya vs Jev — comparação prática de System One Models

Este notebook compara **Laya** e **Jev** executando a mesma tarefa de decisão estruturada.

A proposta é deliberadamente simples:

1. definir um mesmo `state`;
2. definir três decisões equivalentes:
   - `choice`: qual departamento deve atender;
   - `score`: qual o nível de urgência;
   - `noul`: existe risco de cancelamento?;
3. executar primeiro no **Laya**;
4. executar depois no **Jev**;
5. normalizar as respostas em uma tabela para facilitar a comparação.

---

## Ideia central

Os dois sistemas seguem praticamente a mesma abstração:

```text
estado não estruturado / estruturado
              │
              ▼
      perguntas tipadas
              │
      ┌───────┼────────┐
      ▼       ▼        ▼
    choice   score    noul
      │       │        │
      └───────┼────────┘
              ▼
     decisões probabilísticas
```

A principal diferença neste notebook não é **o tipo de problema** que eles resolvem, mas **como são executados e como a API é expressa**.

- **Laya**: modelo open-source executado localmente, normalmente via `Router`.
- **Jev**: serviço da TypeSafe acessado por API, usando classes tipadas no SDK Python.



## 1. Observação sobre hardware

O Laya roda localmente e precisa carregar um checkpoint na máquina. Isso implica:

- download inicial do modelo;
- uso de memória RAM/VRAM;
- latência dependente do seu hardware.

O Jev não carrega o modelo localmente: a inferência é feita pela API da TypeSafe.


In [1]:
from __future__ import annotations

import os
from pprint import pprint
from time import perf_counter

import pandas as pd
from dotenv import load_dotenv

from laya import Router

from typesafe_sdk import (
    TypeSafeClient,
    Choice,
    Score,
    Noul,
)

load_dotenv()


True


# 2. Mesmo problema para os dois modelos

Vamos simular um ticket de suporte.

O cliente:

- informa uma cobrança duplicada;
- pede reembolso;
- exige resolução rápida;
- ameaça cancelar o plano.

Isso permite avaliar diferentes tipos de decisão no mesmo estado.


In [2]:

state = {
    "sender": "cliente@exemplo.com",
    "subject": "Cobrança duplicada na assinatura",
    "body": (
        "Fui cobrado duas vezes pela assinatura deste mês. "
        "Preciso que o valor duplicado seja estornado hoje. "
        "Se isso não for resolvido rapidamente, vou cancelar meu plano."
    ),
}

pprint(state)


{'body': 'Fui cobrado duas vezes pela assinatura deste mês. Preciso que o '
         'valor duplicado seja estornado hoje. Se isso não for resolvido '
         'rapidamente, vou cancelar meu plano.',
 'sender': 'cliente@exemplo.com',
 'subject': 'Cobrança duplicada na assinatura'}



# 3. Semântica das perguntas

Usaremos as mesmas três decisões nos dois modelos.

## `department` — Choice

Escolhe **uma alternativa** entre opções predefinidas.

```text
billing
technical
sales
other
```

## `urgency` — Score

Posiciona o caso em uma escala ordinal:

```text
not urgent → soon → critical
```

## `churn_risk` — Noul

Pergunta binária probabilística:

```text
O cliente ameaça cancelar ou deixar o serviço?
```

Um `noul` próximo de `1` indica forte probabilidade de `True`.



# 4. Laya

O Laya representa as perguntas usando **dicionários Python**.

A estrutura básica é:

```python
{
    "nome_da_pergunta": {
        "type": "...",
        "instructions": "...",
        "criteria": ...
    }
}
```

Isso é bastante próximo de um schema JSON.

O `Router` é a interface recomendada porque pode selecionar automaticamente o checkpoint adequado ao idioma.


In [3]:

laya_questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this customer request?",
        "criteria": {
            "billing": "Invoices, payments, duplicate charges and refunds",
            "technical": "Software bugs, outages and technical problems",
            "sales": "Pricing, upgrades and new contracts",
            "other": "Requests that do not belong to the other departments",
        },
    },

    "urgency": {
        "type": "score",
        "instructions": "How urgent is this customer request?",
        "criteria": [
            "not urgent",
            "should be handled soon",
            "critical, blocking or requires immediate attention",
        ],
    },

    "churn_risk": {
        "type": "noul",
        "instructions": (
            "Does the customer explicitly threaten to cancel, "
            "leave, or stop using the service?"
        ),
    },
}

pprint(laya_questions)


{'churn_risk': {'instructions': 'Does the customer explicitly threaten to '
                                'cancel, leave, or stop using the service?',
                'type': 'noul'},
 'department': {'criteria': {'billing': 'Invoices, payments, duplicate charges '
                                        'and refunds',
                             'other': 'Requests that do not belong to the '
                                      'other departments',
                             'sales': 'Pricing, upgrades and new contracts',
                             'technical': 'Software bugs, outages and '
                                          'technical problems'},
                'instructions': 'Which department should handle this customer '
                                'request?',
                'type': 'choice'},
 'urgency': {'criteria': ['not urgent',
                          'should be handled soon',
                          'critical, blocking or requires immediate attention'


## 4.1 Inicializar o Router do Laya

```python
router = Router()
```

O `Router` pode selecionar o checkpoint conforme o idioma/script.

Para este exemplo em português, podemos forçar explicitamente:

```python
model="multilingual"
```

Isso torna o experimento mais previsível e evita depender da heurística de roteamento.


In [4]:

laya_router = Router()



## 4.2 Fazer a inferência com Laya

Uma única chamada recebe:

- o `state`;
- todas as perguntas;
- o checkpoint desejado.

O Laya processa as decisões como uma tarefa estruturada, sem gerar texto livre.


In [5]:

laya_start = perf_counter()

laya_result = laya_router.predict(
    state,
    laya_questions,
    model="multilingual",
)

laya_elapsed = perf_counter() - laya_start

pprint(laya_result)


/home/leonardo/Documentos/github/jev_laya_studies/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|█| 5/5 [00:00<00:00, 2


{'answers': {'churn_risk': {'action': {'act_probability': 1.0},
                            'answer_confidence': 0.9889,
                            'confidence': 0.9889,
                            'noul': 0.0111,
                            'type': 'noul'},
             'department': {'action': {'act_probability': 1.0},
                            'answer_confidence': 0.9996,
                            'choice': 'billing',
                            'confidence': 0.9968,
                            'probabilities': {'billing': 0.9996,
                                              'other': 0.0001,
                                              'sales': 0.0002,
                                              'technical': 0.0001},
                            'type': 'choice'},
             'urgency': {'action': {'act_probability': 1.0},
                         'answer_confidence': 0.945,
                         'confidence': 0.8037,
                         'legend': {'0': 'not urgent'


## 4.3 Inspecionar respostas do Laya

O retorno do Laya é um dicionário.

Por isso acessamos respostas usando chaves como:

```python
result["answers"]["department"]
```

e também podemos observar metadados de roteamento:

```python
result["routing"]
```


In [6]:

print("=== LAYA ===")

print(
    "Department:",
    laya_result["answers"]["department"],
)

print(
    "Urgency:",
    laya_result["answers"]["urgency"],
)

print(
    "Churn risk:",
    laya_result["answers"]["churn_risk"],
)

print(
    "Routing:",
    laya_result.get("routing"),
)

print(
    f"Elapsed: {laya_elapsed:.4f} s"
)


=== LAYA ===
Department: {'type': 'choice', 'choice': 'billing', 'probabilities': {'billing': 0.9996, 'technical': 0.0001, 'sales': 0.0002, 'other': 0.0001}, 'confidence': 0.9968, 'answer_confidence': 0.9996, 'action': {'act_probability': 1.0}}
Urgency: {'type': 'score', 'score': 1.9445, 'legend': {'0': 'not urgent', '1': 'should be handled soon', '2': 'critical, blocking or requires immediate attention'}, 'probabilities': {'0': 0.0004, '1': 0.0546, '2': 0.945}, 'confidence': 0.8037, 'answer_confidence': 0.945, 'action': {'act_probability': 1.0}}
Churn risk: {'type': 'noul', 'noul': 0.0111, 'confidence': 0.9889, 'answer_confidence': 0.9889, 'action': {'act_probability': 1.0}}
Routing: {'model': 'multilingual', 'repo': 'convaiinnovations/laya/multilingual', 'reason': "explicit model='multilingual'", 'detection': None, 'workflow': None}
Elapsed: 10.8950 s



# 5. Jev

No Jev, a abstração é praticamente a mesma, mas o SDK Python usa **classes tipadas**.

Em vez de:

```python
{
    "type": "choice",
    ...
}
```

escrevemos:

```python
Choice(...)
```

Em vez de:

```python
{
    "type": "score",
    ...
}
```

escrevemos:

```python
Score(...)
```

E, para uma decisão booleana probabilística:

```python
Noul(...)
```

Essa é uma diferença principalmente de **interface do SDK**, não de conceito.


In [7]:

jev_questions = {
    "department": Choice(
        instructions=(
            "Which department should handle this customer request?"
        ),
        criteria={
            "billing": "Invoices, payments, duplicate charges and refunds",
            "technical": "Software bugs, outages and technical problems",
            "sales": "Pricing, upgrades and new contracts",
            "other": "Requests that do not belong to the other departments",
        },
    ),

    "urgency": Score(
        instructions="How urgent is this customer request?",
        criteria=[
            "not urgent",
            "should be handled soon",
            "critical, blocking or requires immediate attention",
        ],
    ),

    "churn_risk": Noul(
        instructions=(
            "Does the customer explicitly threaten to cancel, "
            "leave, or stop using the service?"
        ),
    ),
}



## 5.1 Executar a mesma tarefa no Jev

A chamada equivalente é:

```python
client.system_one(
    state=state,
    questions=questions,
)
```

Ao contrário do Laya, não precisamos carregar um checkpoint local.

É necessário apenas que `TYPESAFE_API_KEY` esteja disponível no ambiente.


In [8]:

assert os.getenv("TYPESAFE_API_KEY"), (
    "Defina TYPESAFE_API_KEY no arquivo .env antes de executar esta célula."
)

jev_start = perf_counter()

with TypeSafeClient() as jev_client:
    jev_result = jev_client.system_one(
        state=state,
        questions=jev_questions,
    )

jev_elapsed = perf_counter() - jev_start

jev_result


SystemOneResponse(model='jev-1.13.0', usage=Usage(input_tokens=504, output_tokens=79), answers={'department': ChoiceAnswer(type='choice', choice='billing', confidence=1.0, probabilities={'other': 0.0, 'sales': 0.0, 'billing': 1.0, 'technical': 0.0}), 'urgency': ScoreAnswer(type='score', score=1.95, confidence=0.92, legend={0: 'not urgent', 1: 'should be handled soon', 2: 'critical, blocking or requires immediate attention'}, probabilities={0: 0.0, 1: 0.05, 2: 0.95}), 'churn_risk': NoulAnswer(type='noul', noul=0.97)})


## 5.2 Inspecionar respostas do Jev

Uma diferença de ergonomia aparece aqui.

No Laya:

```python
result["answers"]["churn_risk"]["noul"]
```

No Jev:

```python
result.answers["churn_risk"].noul
```

Ou seja:

- Laya retorna estruturas baseadas principalmente em dicionários;
- Jev retorna objetos tipados do SDK.


In [9]:

print("=== JEV ===")

print(
    "Department:",
    jev_result.answers["department"],
)

print(
    "Urgency:",
    jev_result.answers["urgency"],
)

print(
    "Churn risk:",
    jev_result.answers["churn_risk"],
)

print(
    f"Elapsed: {jev_elapsed:.4f} s"
)


=== JEV ===
Department: type='choice' choice='billing' confidence=1.0 probabilities={'other': 0.0, 'sales': 0.0, 'billing': 1.0, 'technical': 0.0}
Urgency: type='score' score=1.95 confidence=0.92 legend={0: 'not urgent', 1: 'should be handled soon', 2: 'critical, blocking or requires immediate attention'} probabilities={0: 0.0, 1: 0.05, 2: 0.95}
Churn risk: type='noul' noul=0.97
Elapsed: 0.5851 s



# 6. Comparar o Noul diretamente

O `Noul` é o caso mais simples de comparar porque nos dois sistemas ele representa essencialmente:

```text
P(True | state, question)
```

Assim, para `churn_risk`, podemos colocar os dois valores lado a lado.


In [10]:

laya_churn = laya_result[
    "answers"
]["churn_risk"]["noul"]

jev_churn = jev_result.answers[
    "churn_risk"
].noul

noul_comparison = pd.DataFrame(
    [
        {
            "model": "Laya",
            "churn_risk_probability": laya_churn,
        },
        {
            "model": "Jev",
            "churn_risk_probability": jev_churn,
        },
    ]
)

noul_comparison


,model,churn_risk_probability
0,Laya,0.0111
1,Jev,0.9700



# 7. Normalização das respostas

Os dois sistemas possuem conceitos muito semelhantes, porém seus objetos de retorno não são idênticos.

Para experimentos maiores, é útil criar uma pequena **camada adaptadora** que converta ambos para um formato comum.

Isso permite escrever código de benchmark sem acoplá-lo ao SDK de um modelo específico.


In [11]:

def normalize_laya_result(result: dict) -> dict:
    answers = result["answers"]

    return {
        "department": answers["department"].get("choice"),
        "department_confidence": answers["department"].get("confidence"),

        # Dependendo da versão do Laya, o score pode trazer
        # campos adicionais como distribuição/confidence.
        "urgency": answers["urgency"].get("score"),
        "urgency_confidence": answers["urgency"].get("confidence"),

        "churn_risk": answers["churn_risk"].get("noul"),

        "routing_model": (
            result.get("routing", {})
            .get("model")
        ),
    }


def normalize_jev_result(result) -> dict:
    department = result.answers["department"]
    urgency = result.answers["urgency"]
    churn = result.answers["churn_risk"]

    return {
        # Os atributos exatos disponíveis além da decisão principal
        # podem variar conforme a versão do SDK.
        "department": getattr(department, "choice", None),
        "department_confidence": getattr(
            department,
            "confidence",
            None,
        ),

        "urgency": getattr(urgency, "score", None),
        "urgency_confidence": getattr(
            urgency,
            "confidence",
            None,
        ),

        "churn_risk": getattr(churn, "noul", None),

        "routing_model": None,
    }


In [12]:

laya_normalized = normalize_laya_result(
    laya_result
)

jev_normalized = normalize_jev_result(
    jev_result
)

comparison = pd.DataFrame(
    [
        {
            "model": "Laya",
            **laya_normalized,
            "elapsed_seconds": laya_elapsed,
        },
        {
            "model": "Jev",
            **jev_normalized,
            "elapsed_seconds": jev_elapsed,
        },
    ]
)

comparison


,model,department,department_confidence,urgency,urgency_confidence,churn_risk,routing_model,elapsed_seconds
0,Laya,billing,0.9968,1.9445,0.8037,0.0111,multilingual,10.894965
1,Jev,billing,1.0000,1.9500,0.9200,0.9700,NaN,0.585104



# 8. Semelhanças

## 8.1 Mesmo paradigma

Laya e Jev recebem:

```text
STATE + TYPED QUESTIONS
```

e devolvem:

```text
TYPED PROBABILISTIC DECISIONS
```

Nenhum deles foi desenhado primariamente para produzir respostas textuais longas.

---

## 8.2 Primitivas equivalentes

| Conceito | Laya | Jev |
|---|---|---|
| Selecionar uma opção | `type="choice"` | `Choice(...)` |
| Avaliar uma escala ordinal | `type="score"` | `Score(...)` |
| Probabilidade booleana | `type="noul"` | `Noul(...)` |

---

## 8.3 Várias decisões sobre o mesmo estado

Nos dois casos podemos fazer:

```text
                        ticket
                           │
             ┌─────────────┼─────────────┐
             ▼             ▼             ▼
         department      urgency      churn_risk
          Choice          Score          Noul
```

Isso é importante porque evita transformar todo o workflow em um único prompt monolítico.

---

## 8.4 Política continua em Python

Os modelos podem informar probabilidades.

A aplicação continua responsável por decidir o que fazer:

```python
if churn_probability >= 0.80:
    escalate_to_retention_team()
```

Isso mantém a lógica de negócio fora do modelo.



# 9. Diferenças

## 9.1 Execução

### Laya

```text
Python
  │
  ▼
checkpoint Hugging Face
  │
  ▼
CPU / GPU local
```

O modelo roda localmente.

### Jev

```text
Python
  │
  ▼
TypeSafe API
  │
  ▼
Jev
```

A inferência ocorre remotamente.

---

## 9.2 Distribuição

**Laya**

- open-source;
- checkpoints disponíveis no Hugging Face;
- pode ser executado offline depois que o checkpoint estiver disponível localmente;
- você controla o hardware de inferência.

**Jev**

- serviço disponibilizado pela TypeSafe;
- acesso via API;
- infraestrutura de inferência gerenciada pelo provedor.

---

## 9.3 Forma das perguntas

Laya usa schemas semelhantes a JSON:

```python
{
    "type": "noul",
    "instructions": "..."
}
```

Jev usa objetos:

```python
Noul(
    instructions="..."
)
```

---

## 9.4 Roteamento no Laya

O Laya possui um `Router` explícito capaz de selecionar checkpoints como:

```text
english
multilingual
typed-decisions
```

Por isso o retorno também inclui metadados de routing.

No exemplo deste notebook usamos:

```python
model="multilingual"
```

porque o ticket está em português.

---

## 9.5 Custos e recursos

Com Laya, o custo dominante é o seu próprio hardware:

```text
RAM / VRAM + CPU/GPU + energia + operação
```

Com Jev, o custo dominante é a chamada ao serviço.

Portanto, simplesmente comparar `elapsed_seconds` deste notebook **não constitui um benchmark rigoroso**: um caminho é local e o outro inclui rede.



# 10. Comparação conceitual

Podemos resumir a relação desta forma:

```text
                 System One decision
                         │
                ┌────────┴────────┐
                │                 │
                ▼                 ▼
              Laya               Jev
                │                 │
        open-source/local      API/cloud
                │                 │
        dict questions       typed classes
                │                 │
          Router.predict     system_one
                │                 │
                └────────┬────────┘
                         │
                   Choice / Score /
                       Noul
```

Isso explica por que é relativamente fácil escrever uma camada adaptadora entre os dois.



# 11. Segundo experimento: somente Noul

Para uma comparação ainda mais direta, podemos retirar `Choice` e `Score` e fazer várias perguntas booleanas idênticas.

Isso é útil para testar concordância entre os modelos.


In [13]:

binary_questions = {
    "refund_requested": (
        "Does the customer explicitly request a refund or reimbursement?"
    ),
    "duplicate_charge": (
        "Does the customer report being charged more than once "
        "for the same billing period?"
    ),
    "churn_risk": (
        "Does the customer explicitly threaten to cancel, leave, "
        "or stop using the service?"
    ),
    "urgent": (
        "Does the customer communicate that the issue requires "
        "immediate or very fast resolution?"
    ),
}


In [14]:

laya_binary_questions = {
    name: {
        "type": "noul",
        "instructions": instruction,
    }
    for name, instruction
    in binary_questions.items()
}

laya_binary_result = laya_router.predict(
    state,
    laya_binary_questions,
    model="multilingual",
)

laya_binary_scores = {
    name: laya_binary_result[
        "answers"
    ][name]["noul"]
    for name in binary_questions
}

laya_binary_scores


{'refund_requested': 0.9809,
 'duplicate_charge': 0.9973,
 'churn_risk': 0.0108,
 'urgent': 0.9635}

In [15]:

jev_binary_questions = {
    name: Noul(
        instructions=instruction
    )
    for name, instruction
    in binary_questions.items()
}

with TypeSafeClient() as jev_client:
    jev_binary_result = jev_client.system_one(
        state=state,
        questions=jev_binary_questions,
    )

jev_binary_scores = {
    name: jev_binary_result.answers[
        name
    ].noul
    for name in binary_questions
}

jev_binary_scores


{'refund_requested': 0.97,
 'duplicate_charge': 0.98,
 'churn_risk': 0.97,
 'urgent': 0.97}

In [16]:

binary_comparison = pd.DataFrame(
    {
        "question": list(
            binary_questions.keys()
        ),
        "Laya": [
            laya_binary_scores[name]
            for name in binary_questions
        ],
        "Jev": [
            jev_binary_scores[name]
            for name in binary_questions
        ],
    }
)

binary_comparison["absolute_difference"] = (
    binary_comparison["Laya"]
    - binary_comparison["Jev"]
).abs()

binary_comparison


,question,Laya,Jev,absolute_difference
0,refund_requested,0.9809,0.97,0.0109
1,duplicate_charge,0.9973,0.98,0.0173
2,churn_risk,0.0108,0.97,0.9592
3,urgent,0.9635,0.97,0.0065
